# Customer Churn Prediction from Scratch
 Open this notebook from the project directory and run cells in order. Core experiments use only the Python standard library.

In [1]:
import random
import experiments
from trees import DecisionTree

SEED = 3411
random.seed(SEED)
datasets = experiments.load_datasets()
results = {"seed": SEED, "baseline": experiments.baseline_metrics(datasets)}

## 2.1 Data, reproducibility and baseline

The target is churned, with 1 representing churn within 30 days. Customer ID and target are excluded from the 15 inputs. Training labels determine the majority baseline for every split. Every randomized experiment uses master seed 3411; tree seeds are deterministically derived from it. The completed methods replace the starter TODOs: information gain, numerical threshold selection and ensemble experiment drivers.

In [2]:
experiments.print_summary(datasets)

Random seed: 3411
Majority class from training: 0
train: rows=840, class counts={0: 520, 1: 320}
validation: rows=180, class counts={0: 111, 1: 69}
test: rows=180, class counts={0: 116, 1: 64}
Baseline train: accuracy=0.6190, F1=0.0000
Baseline validation: accuracy=0.6167, F1=0.0000
Baseline test: accuracy=0.6444, F1=0.0000
Implemented: information gain, midpoint search and all ensemble experiments.


## 2.2 Categorical splits and depth control

This section uses categorical inputs only. Leaves predict the majority label, breaking ties in favour of 0. A pure node, a configured depth limit or absence of positive information gain stops growth. All reported accuracies are printed beside the corresponding majority-class baseline and positive-class F1.

In [3]:
results["depths"] = experiments.run_depths(datasets)

Maximum depth allowed: 2
Actual depth: 2
Internal nodes: 4
train: accuracy=0.6583, F1=0.3608; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.0393
validation: accuracy=0.6333, F1=0.3654; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0167
test: accuracy=0.7111, F1=0.4694; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0667

Maximum depth allowed: 3
Actual depth: 3
Internal nodes: 14
train: accuracy=0.7024, F1=0.5675; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.0833
validation: accuracy=0.6000, F1=0.4375; baseline accuracy=0.6167, F1=0.0000; accuracy difference=-0.0167
test: accuracy=0.6889, F1=0.5333; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0444

Maximum depth allowed: 4
Actual depth: 4
Internal nodes: 43
train: accuracy=0.7417, F1=0.6316; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.1226
validation: accuracy=0.5944, F1=0.4160; baseline accuracy=0.6167, F1=0.0000; accuracy difference=-0.0222
test: 

I choose maximum depth 2 using validation accuracy as the primary criterion. It gives validation accuracy 0.6333 and F1 0.3654, compared with baseline accuracy 0.6167 and F1 0.0000, and has only four internal nodes. Depth 3 has a better validation F1 of 0.4375 but lower accuracy, 0.6000. The unrestricted tree improves training performance while validation performance falls, suggesting overfitting. Test results were not used to choose the depth. Hidden subtrees in the printed tree are explicitly labelled rather than presented as real leaves.

## 2.3 Numerical threshold search

From this section onward, numerical and categorical features are considered together. Every consecutive-distinct-value midpoint is evaluated. The root thresholds below use all training rows; thresholds in a later branch use only the rows reaching that branch.

In [4]:
results["thresholds"] = experiments.run_thresholds(datasets)

tenure_months: threshold=10.5000, information gain=0.076829
monthly_charges: threshold=71.4100, information gain=0.006799
avg_monthly_gb: threshold=40.2500, information gain=0.006151
support_tickets_90d: threshold=1.5000, information gain=0.006915
Tree using categorical and numerical attributes:
contract_type
  [Month-to-month] tenure_months <= 6.500
  [Month-to-month]   [T] Predict 1
  [Month-to-month]   [F] Predict 0
  [One year] late_payments_6m <= 3.500
  [One year]   [T] Predict 0
  [One year]   [F] Predict 1
  [Two year] tenure_months <= 32.500
  [Two year]   [T] Predict 0
  [Two year]   [F] Predict 0


Numerical search evaluates many candidate thresholds after sorting, while a categorical feature is grouped and evaluated once. A simple implementation rescans all n rows at every candidate, costing O(qn+n log n) for q thresholds. This project evaluates exactly the same midpoints with cumulative label counts, reducing the numerical search to O(n log n) per feature for binary labels. Sorting and checking many tests still require work that a single categorical grouping does not.

In the learned tree, contract type is selected at the root. Month-to-month customers with tenure at most 6.5 months are predicted to churn. One-year customers with at least four late payments in six months are predicted to churn. Among the four numerical features tested at the root, tenure has the largest information gain. These are predictive associations, not causal claims.

## 2.4.1 Bagging

Each tree uses a training bootstrap sample of the original sample size, sampled with replacement. Predictions use majority voting, with ties resolved as class 0. B is tested at 5, 10 and 20. All features are available at every node.

In [5]:
results["bagging"], bagging_models = experiments.run_bagging(datasets)

Bagging: B = 5
train: accuracy=0.9536, F1=0.9378; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3345
validation: accuracy=0.6611, F1=0.5041; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0444
test: accuracy=0.6556, F1=0.5303; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0111
Average tree depth: 12.00

Bagging: B = 10
train: accuracy=0.9714, F1=0.9614; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3524
validation: accuracy=0.6778, F1=0.4912; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0611
test: accuracy=0.6778, F1=0.5167; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0333
Average tree depth: 12.70

Bagging: B = 20
train: accuracy=0.9929, F1=0.9906; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3738
validation: accuracy=0.6833, F1=0.5128; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0667
test: accuracy=0.7167, F1=0.5984; baseline accuracy=0.6444, F1=0.0000; accuracy differ

## 2.4.2 Random forests

The bootstrap mechanism is identical to bagging. Each node independently samples m candidate features. With 15 inputs, floor(sqrt(15))=3; m=15 uses all inputs. The same master seed, tree count and bootstrap generator make m=15 an exact comparison with bagging.

In [6]:
results["forests"] = experiments.run_forests(datasets, bagging_models)

Random forest: B=10, m=3, d=15
train: accuracy=0.9655, F1=0.9527; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3464
validation: accuracy=0.6333, F1=0.4211; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0167
test: accuracy=0.7000, F1=0.5000; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0556
Average tree depth: 14.50

Random forest: B=10, m=15, d=15
train: accuracy=0.9714, F1=0.9614; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3524
validation: accuracy=0.6778, F1=0.4912; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0611
test: accuracy=0.6778, F1=0.5167; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0333
Average tree depth: 12.70

m=d predictions match Bagging: {'train': True, 'validation': True, 'test': True}


With m=3, validation accuracy is 0.6333 and F1 is 0.4211; with m=15, they are 0.6778 and 0.4912. Both exceed the validation baseline accuracy 0.6167 and F1 0.0000. The all-feature predictions match bagging on every split. Random feature selection can reduce correlation between trees, so voting can reduce variance. However, it can exclude useful features at a node. Here m=3 does not improve validation accuracy or F1 over bagging; randomness is not guaranteed to improve every dataset or metric.

## 2.4.3 Ensemble design: validation-weighted shallow trees

I use 20 bootstrap trees with maximum depth 4 and entropy-based information gain. Each tree receives a non-negative vote weight equal to its validation accuracy. The class with the largest total weight is predicted, with ties resolved as 0. If every weight is zero, voting falls back to equal weights. The identical shallow trees with equal votes provide a control. I also evaluate a single mixed-feature tree of depth 3 as an interpretable comparison.

The weights use validation labels only; test labels never affect model fitting or weighting. Because validation data choose the weights, the validation score is a tuning result and may be optimistic. The independent test score is reported for evaluation. Depth 4 and 20 trees were fixed for this design rather than chosen by looking at test results.

In [7]:
results["weighted"] = experiments.run_weighted(datasets)

Vote weights (validation accuracy): [0.6722, 0.6722, 0.6278, 0.6778, 0.6667, 0.6444, 0.6278, 0.6778, 0.65, 0.6778, 0.6889, 0.7111, 0.6722, 0.6611, 0.6111, 0.6611, 0.6333, 0.6111, 0.65, 0.6389]
train: accuracy=0.7488, F1=0.5577; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.1298
validation: accuracy=0.6944, F1=0.4762; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0778
test: accuracy=0.7278, F1=0.5149; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0833
Average tree depth: 4.00

train: accuracy=0.7536, F1=0.5767; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.1345
validation: accuracy=0.7000, F1=0.5000; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0833
test: accuracy=0.7333, F1=0.5294; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0889
Average tree depth: 4.00

train: accuracy=0.7238, F1=0.5451; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.1048
validation: accuracy=0.6500, F1=0.4522; baselin

The weighted model has validation accuracy 0.7000 and F1 0.5000, compared with 0.6944 and 0.4762 for equal voting on the same shallow trees. The validation baseline is accuracy 0.6167 and F1 0.0000. Limiting tree depth can reduce very specific rules for small customer groups. Weighting tries to give more influence to trees that generalise better. This may help when churn depends on several interacting signals such as contract type, tenure and payment behaviour, but similar validation accuracies can also produce little change.

For interpretability, I would use the single mixed-feature depth-3 tree as the clearest trade-off: it has 10 internal nodes and validation accuracy 0.6500, F1 0.4522 (baseline 0.6167 and 0.0000). Its rules can be inspected directly. An ensemble can be preferable when predictive performance matters more than a short explanation. This is a practical preference, not proof that one model is universally best.

## 2.4.4 Early stopping

My rule stops a node before split search if it contains fewer than 20 training examples. Otherwise it accepts a split only when information gain is at least 0.01 bits and strictly positive. The same rule is applied to bagging and random forests. The control uses the original stopping rules (minimum two examples, strictly positive gain). Both settings use 20 trees, the same master seed, the same bootstrap samples and no maximum depth.

The rule aims to avoid learning unstable rules from very small customer groups and splits that reduce little uncertainty. It can reduce variance and training cost, but may lose useful rare-churn patterns. Thresholds were fixed in advance. Training time below measures fit only, excludes evaluation and is hardware-dependent; one timed run is descriptive rather than a rigorous benchmark.

In [8]:
results["early_stopping"] = experiments.run_early_stopping(datasets)

bagging: early stopping = False
Training time: 0.780871 seconds
train: accuracy=0.9929, F1=0.9906; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3738
validation: accuracy=0.6833, F1=0.5128; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0667
test: accuracy=0.7167, F1=0.5984; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0722
Average tree depth: 12.90

bagging: early stopping = True
Training time: 0.630116 seconds
train: accuracy=0.8738, F1=0.8153; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.2548
validation: accuracy=0.6444, F1=0.4667; baseline accuracy=0.6167, F1=0.0000; accuracy difference=+0.0278
test: accuracy=0.7056, F1=0.5620; baseline accuracy=0.6444, F1=0.0000; accuracy difference=+0.0611
Average tree depth: 10.75

random_forest: early stopping = False
Training time: 0.227354 seconds
train: accuracy=0.9881, F1=0.9841; baseline accuracy=0.6190, F1=0.0000; accuracy difference=+0.3690
validation: accuracy=0.6278, F1=0.4274; b

Bagging: training time changes from 0.7809s to 0.6301s, and average depth from 12.90 to 10.75. On validation, accuracy changes from 0.6833 to 0.6444 (difference -0.0389); F1 changes from 0.5128 to 0.4667 (difference -0.0462). The baseline is accuracy 0.6167, F1 0.0000. On test, accuracy changes from 0.7167 to 0.7056 (difference -0.0111); F1 changes from 0.5984 to 0.5620 (difference -0.0364). The baseline is accuracy 0.6444, F1 0.0000.

Random forest: training time changes from 0.2274s to 0.1295s, and average depth from 14.20 to 9.45. On validation, accuracy changes from 0.6278 to 0.6333 (difference +0.0056); F1 changes from 0.4274 to 0.2326 (difference -0.1948). The baseline is accuracy 0.6167, F1 0.0000. On test, accuracy changes from 0.7000 to 0.6944 (difference -0.0056); F1 changes from 0.5345 to 0.3038 (difference -0.2307). The baseline is accuracy 0.6444, F1 0.0000.

These results show the measured trade-off; an improvement in one metric does not guarantee improvement in another. I report the test results without using them to alter the stopping rule.

In this run, the bagging rule reduces both validation and test accuracy and F1. For the random forest, validation accuracy rises slightly, but validation F1 falls sharply; both test accuracy and test F1 fall. I would not select this particular rule when churn F1 is the main objective, despite the smaller trees and shorter measured training times. This conclusion comes from the validation comparison; test metrics are reported without retuning the rule.


In [9]:
experiments.save_results(results)
print("Saved results/results.json and results/metrics.csv")

Saved results/results.json and results/metrics.csv
